<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 02 · 项目规则变了，记忆怎么办

上线前，团队把批量导入上限从 100 行改成了 200 行。旧约定曾经是对的，如今却可能误导后续工作。
我们希望新的会话使用新规则，同时仍能回答“当时依据的是哪个版本”。

这一篇从一次修订开始，接着模拟两个人同时修改，最后让一条过期约定退出日常召回。

**这一篇的收获：** 修订和遗忘 Atomic Memory，读取历史版本，区分正文 revision 与生命周期 state_version，并用 ETag 防止并发覆盖。

**运行准备：** 从 [教程入口](README.md) 安装依赖并启动 Jupyter。每篇都带有自己的数据，可以独立运行。
本篇不需要模型或 API Key。
建议先逐格运行，读完输出再继续；完整重跑时使用 **Restart Kernel & Run All**。

## 先准备一个自己的实验空间

下面的辅助代码只负责启动本地 Server、建立 Client 和整理输出。默认每次完整运行使用新的 SQLite 数据库；选择 OceanBase 时，使用专用测试库并为本次实验创建新的 Scope。
后端设置见 [README](README.md#使用-oceanbase-运行)。关键的写入、检索、审核与交接调用会直接写在后面的单元格里。


In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))

if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("02", features=())
client = lab.client
assert client is not None

现在创建本篇的项目 Scope。`title` 是给人看的名称，真正用于调用的是 Server 返回的 `scope_id`。
你可以改变标题；不要自己根据标题或目录拼出一个 Scope ID。


In [ ]:
from powercontext.http import CreateScopeRequest

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 02",
        summary="第 02 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-02",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 建立最初的约定

本篇会重新创建自己的数据，不需要上一篇的运行状态。先保存 100 行的上限，并保留响应里的记忆记录和精确 Artifact 引用。

每条记忆都是独立的 `atomic-memory` Artifact。`artifact_id` 标识这条记忆，`revision` 标识不可变的正文版本；
`state` 和 `state_version` 描述当前生命周期，不属于历史正文。

In [ ]:
from powercontext.http import (
    AtomicMemoryInput,
    AtomicMemoryLifecycleRequest,
    AtomicMemoryWriteContent,
    ListArtifactRevisionsRequest,
    ListArtifactsRequest,
    ListMemoryEntriesRequest,
    RememberMemoryRequest,
    ReplaceAtomicMemoryArtifactRequest,
    SearchMemoryRequest,
)

original = await client.remember_memory(
    RememberMemoryRequest(
        scope_id=scope_id,
        kind="constraint",
        text="batch_limit: 每批最多导入 100 行。",
        reason="首期导入约定",
    )
)
assert original.changed and len(original.records) == 1
original_memory = original.records[0]
original_ref = original_memory.artifact
assert original_ref.family == "atomic-memory" and original_memory.state == "active"
show(original_memory)
memory_page = await client.list_artifacts(scope_id, original_ref.family, ListArtifactsRequest())
assert any(item.artifact_id == original_ref.artifact_id for item in memory_page.items)
table([{"制品": item.artifact_id, "当前版本": item.revision} for item in memory_page.items])

## 2. 根据读到的版本进行修订

修订使用通用 `replace_artifact`，提交完整正文，并用正文读取响应的 ETag 作为 `expected_etag`。
Python Client 的普通制品读取结果不暴露响应头，因此这里用一次真实 HTTP GET 同时读取正文和 ETag；不要自行拼接 ETag。
成功后，Artifact ID 不变，正文 revision 增加，当前生命周期保持不变。

In [ ]:
import httpx

artifact_path = f"/v1/scopes/{scope_id}/artifacts/{original_ref.family}/{original_ref.artifact_id}"
async with httpx.AsyncClient(base_url=lab.base_url, trust_env=False) as http:
    response = await http.get(artifact_path)
    response.raise_for_status()
    original_etag = response.headers["etag"]
    assert response.json()["revision"] == original_ref.revision
    assert response.json()["content"]["text"] == original_memory.text

revised_text = "batch_limit: 每批最多导入 200 行；超过时提示拆分文件。"
revised = await client.replace_artifact(
    scope_id,
    original_ref.family,
    original_ref.artifact_id,
    ReplaceAtomicMemoryArtifactRequest(content=AtomicMemoryWriteContent(kind="constraint", text=revised_text)),
    expected_etag=original_etag,
)
latest_state = await client.get_atomic_memory_state(scope_id, original_ref.artifact_id)
latest_ref = latest_state.artifact
table([
    {"Revision": original_ref.revision, "制品": original_ref.artifact_id, "内容": original_memory.text},
    {"Revision": latest_ref.revision, "制品": latest_ref.artifact_id, "内容": revised.content["text"]},
])
assert latest_ref.artifact_id == original_ref.artifact_id
assert latest_ref.revision == revised.revision == original_ref.revision + 1
assert latest_state.state == original_memory.state == "active"
assert latest_state.state_version == original_memory.state_version

## 3. 同事还拿着旧版本，会发生什么

假设另一位同事还没看到 200 行的更新，想把上限改为 150 行。下面故意带上旧正文的 ETag。
冲突是这一步期望看到的结果：它提醒调用方重新读取并判断，而不是悄悄覆盖别人的决定。


In [ ]:
from powercontext.client import ServerResponseError

try:
    await client.replace_artifact(
        scope_id,
        original_ref.family,
        original_ref.artifact_id,
        ReplaceAtomicMemoryArtifactRequest(
            content=AtomicMemoryWriteContent(kind="constraint", text="batch_limit: 每批最多导入 150 行。")
        ),
        expected_etag=original_etag,
    )
except ServerResponseError as error:
    assert error.status_code == 412 and error.code == "revision_conflict"
    show({"HTTP": error.status_code, "结果": "旧版本修改被拒绝；先读取当前正文与 ETag，再决定是否更新。"})
else:
    raise AssertionError("预期收到版本冲突。")

## 4. 当前召回和历史读取各自回答什么

搜索回答“现在可用的约定是什么”。精确读取回答“这个引用当时记录了什么”。
我们同时执行两种读取，观察它们并不矛盾。


In [ ]:
old_snapshot = await client.get_artifact_revision(
    scope_id,
    original_ref.family,
    original_ref.artifact_id,
    original_ref.revision,
)
current = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query="batch_limit", mode="fts"))
assert old_snapshot.content["text"] == original_memory.text
assert any(hit.memory.artifact == latest_ref and hit.memory.text == revised_text for hit in current.hits)
assert all(hit.memory.text != original_memory.text for hit in current.hits)
table([
    {"读取方式": "精确历史引用", "内容": old_snapshot.content["text"]},
    *[{"读取方式": "当前搜索", "内容": hit.memory.text} for hit in current.hits],
])
current_artifact = await client.get_artifact(scope_id, latest_ref.family, latest_ref.artifact_id)
assert current_artifact is not None and current_artifact.revision == latest_ref.revision
assert current_artifact.content["text"] == revised_text
show({"这条记忆的历史版本": old_snapshot.revision, "这条记忆的当前版本": current_artifact.revision})

## 5. 当规则不再适用时，让它退出召回

后来系统采用流式导入，整批行数上限已经失去意义。使用 `change_atomic_memory_lifecycle` 将它设为 `forgotten`，
并带上刚读到的精确 Artifact 引用和 `state_version`，防止修改已经变化的记忆。

遗忘增加状态版本，不增加正文 revision；普通搜索和默认列表不再返回它，管理列表仍可包含它，历史正文也仍可读取。
`forgotten` 可以恢复，和不能直接恢复或编辑的 `retired` 状态不同。

In [ ]:
forgotten = await client.change_atomic_memory_lifecycle(
    AtomicMemoryLifecycleRequest(
        scope_id=scope_id,
        target=AtomicMemoryInput(artifact=latest_ref, state_version=latest_state.state_version),
        state="forgotten",
    )
)
assert forgotten.changed and len(forgotten.records) == 1
forgotten_memory = forgotten.records[0]
assert forgotten_memory.artifact == latest_ref and forgotten_memory.state == "forgotten"
assert forgotten_memory.state_version == latest_state.state_version + 1
active = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id))
history = await client.list_memory_entries(ListMemoryEntriesRequest(scope_id=scope_id, include_inactive=True))
assert all(entry.artifact.artifact_id != latest_ref.artifact_id for entry in active.entries)
assert any(entry == forgotten_memory for entry in history.entries)
after_forgetting = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query="batch_limit", mode="fts"))
assert all(hit.memory.artifact.artifact_id != latest_ref.artifact_id for hit in after_forgetting.hits)
retained = await client.get_artifact_revision(scope_id, latest_ref.family, latest_ref.artifact_id, latest_ref.revision)
assert retained.content["text"] == revised_text
original_retained = await client.get_artifact_revision(
    scope_id, original_ref.family, original_ref.artifact_id, original_ref.revision
)
assert original_retained.content == old_snapshot.content
table([
    {"状态": entry.state, "内容": entry.text, "正文 Revision": entry.artifact.revision, "状态版本": entry.state_version}
    for entry in history.entries
])

## 轮到你：区分正文历史与当前状态

下面列出这条 Artifact 的正文版本，并逐个读取精确历史。修订产生新正文版本，遗忘只改变当前生命周期，
因此这里仍是两个正文版本，不能把状态版本当作第三个正文 revision，也不能将当前状态当作历史正文的一部分。

练习：新增一条 `line_number` 约束，读取正文 ETag 后修订一次，再用它自己的 Artifact ID 查看版本列表。
上面的写入与修订单元格就是可复用的答案骨架。

In [ ]:
versions = await client.list_artifact_revisions(
    scope_id, original_ref.family, original_ref.artifact_id, ListArtifactRevisionsRequest()
)
assert versions.next_cursor is None
assert {item.revision for item in versions.items} == {original_ref.revision, latest_ref.revision}
snapshots = []
for item in versions.items:
    snapshot = await client.get_artifact_revision(
        scope_id, original_ref.family, original_ref.artifact_id, item.revision
    )
    snapshots.append(snapshot)
assert {item.revision: item.content["text"] for item in snapshots} == {
    original_ref.revision: original_memory.text,
    latest_ref.revision: revised_text,
}
table([{"Revision": item.revision, "内容": item.content["text"]} for item in snapshots])
state = await client.get_atomic_memory_state(scope_id, original_ref.artifact_id)
assert state.artifact == latest_ref and state.state == "forgotten"
assert state.state_version == forgotten_memory.state_version
show(state)

## 带着结果离开

你已经把“当前适用的知识”和“曾经记录的历史”区分开了。修订用正文 ETag 防止并发覆盖，遗忘使用精确引用和状态版本并影响 active recall，旧正文版本仍能用于追溯。

下面关闭本篇的 Client 和 Server。实验文件仍留在教程的 `.powercontext/` 子目录，便于检查；
清理方法见 [README](README.md#清理实验数据)。如果在中途停止，请运行这个单元格，或关闭 Kernel。

下一篇：[同时做两个项目，如何避免串台](03_scopes.ipynb)。


In [ ]:
await lab.close()
print("本篇 Server 已关闭。")